# OpenVLA-OFT в LIBERO: что дают камера на запястье и датчики

<a href="https://colab.research.google.com/github/bespatim/vla-manipulation-research/blob/experiments/libero-oft/notebooks/libero_oft_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Продолжение главы 1: те же 10 задач LIBERO-Spatial, те же начальные расстановки (0–2), тот же лимит
220 шагов — но модель **OpenVLA-OFT** (`moojink/openvla-7b-oft-finetuned-libero-spatial`).

## План эксперимента

| № | Условие | Что получает модель | Вопрос |
|---|---|---|---|
| 1 | `image_only` | внешняя камера + текст (запястье и датчики заморожены) | как старая OpenVLA по входам — быстрее ли и лучше ли? |
| 2 | `no_proprio` | 2 камеры + текст (датчики заморожены) | что даёт камера на запястье? |
| 3 | `no_wrist` | внешняя камера + датчики + текст (запястье заморожено) | что дают датчики? |
| 4 | `full` | 2 камеры + датчики + текст | как у авторов (97,6%) — как работают вместе? |

**Ожидания (записаны до запуска):**
- модель отвечает раз в 8 шагов и за один проход, поэтому эпизод пройдёт **в несколько раз быстрее**,
  чем у OpenVLA в главе 1 (0,83 с на каждый шаг);
- в условии `full` успех за 30 попыток будет **выше 76,7%** из главы 1;
- в статье нет раздельной проверки «только камера на запястье» и «только датчики» — это и проверяем;
- если без камеры на запястье станет хуже — модель на неё опирается; если наоборот — камера ей мешает
  (например, потому что базовая OpenVLA училась без неё);
- если датчики (в них есть положение пальцев) помогают — смотрим, исчезают ли «пустые захваты» и
  «замирания» из главы 1.

**Как «убираем» вход.** Авторы выложили только модель, обученную сразу со всеми входами. Поэтому вход
не выкидывается, а **замораживается**: модели всё время подаётся кадр запястья / показания датчиков
с первого шага эпизода. Это проверяет, **насколько обученная модель опирается** на вход. Это не то же
самое, что модель, **обученная без** этого входа: для неё в статье есть только одна цифра — 96,2% на
Spatial с одной камерой (Табл. I).

**Видеокарта.** Подходит и бесплатная **T4**: модель сжимается до 4 бит (как OpenVLA в главе 1),
а вычисления идут во float16 — T4 не поддерживает bfloat16, на котором работает код авторов. На **L4** или
**A100** (*Runtime → Change runtime type*, подписка Colab Pro) скрипт сам выберет путь авторов без
изменений — полную точность bfloat16. Режим записывается в результаты (`precision` в `meta.json`).

## 1. Проверка видеокарты

In [ ]:
import subprocess
name, mem, cc = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader,nounits"], text=True
).strip().split(", ")
print(f"{name}: {int(mem) / 1024:.0f} ГБ, compute capability {cc}")
if int(mem) >= 20000 and float(cc) >= 8.0:
    print("✓ Режим авторов: полная точность bfloat16")
else:
    print("✓ Режим для этой видеокарты: модель сжата до 4 бит, вычисления во float16 "
          "(на L4/A100 будет полная точность, как у авторов)")

## 2. Установка (~7–10 мин)

Отдельное окружение Python 3.10 с версиями авторов OpenVLA-OFT (PyTorch 2.2.0, их версия transformers),
симулятор LIBERO и код OFT.

In [ ]:
BRANCH = "experiments/libero-oft"   # после слияния в main поменять на "main"
PY = "/content/venv-oft/bin/python"
REPO_DIR = "/content/vla-manipulation-research"

%cd /content
!test -d vla-manipulation-research || git clone -q -b {BRANCH} https://github.com/bespatim/vla-manipulation-research.git
!cd vla-manipulation-research && git fetch -q origin && git reset -q --hard origin/{BRANCH} && git log --oneline -1
!test -d LIBERO || git clone -q --depth 1 https://github.com/Lifelong-Robot-Learning/LIBERO.git
# Новые версии PyTorch по умолчанию не загружают numpy-массивы через torch.load — разрешаем явно
!sed -i 's/init_states = torch.load(init_states_path)/init_states = torch.load(init_states_path, weights_only=False)/' LIBERO/libero/libero/benchmark/__init__.py
!test -d openvla-oft || git clone -q --depth 1 https://github.com/moojink/openvla-oft.git

!pip install -q uv
!test -d /content/venv-oft || uv venv -q --python 3.10 /content/venv-oft
!uv pip install -q --python {PY} -e /content/openvla-oft -r {REPO_DIR}/notebooks/requirements-libero-oft.txt
!uv pip install -q --python {PY} --no-deps "bddl==1.0.1"
!{PY} -c "import sys, torch, transformers; print('Python', sys.version.split()[0], '| torch', torch.__version__, '| transformers', transformers.__version__, '| CUDA', torch.cuda.is_available())"

## 3. Первый эпизод (~10–15 мин, из них большая часть — скачивание модели ~15 ГБ)

Условие `full`, задача 0. Видео: **слева** — внешняя камера, **справа** — камера на запястье
(в отличие от главы 1, эту камеру модель теперь получает).

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
icd = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
if not os.path.exists(icd):
    os.makedirs(os.path.dirname(icd), exist_ok=True)
    with open(icd, "w") as f:
        f.write('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}')

SUITE = "libero_spatial"
SCRIPT = f"{REPO_DIR}/scripts/libero_oft_eval.py"
ARGS = f"--libero-root /content/LIBERO --oft-root /content/openvla-oft --suite {SUITE}"
OUT = f"/content/results/oft_{SUITE}"

!cd /content/openvla-oft && {PY} {SCRIPT} {ARGS} --conditions full --tasks 0 --trials 1 --out {OUT}/first

from IPython.display import Video, display
display(Video(f"{OUT}/first/full/{SUITE}_task00_trial00.mp4", embed=True, width=768))

## 4. Основной эксперимент: 4 условия × 10 задач × 3 попытки

120 эпизодов; модель загружается один раз. Порядок — как в плане: `image_only` → `no_proprio` →
`no_wrist` → `full`. Результаты пишутся по ходу: при обрыве сессии сделанное сохранится
(если перед этим подключить Диск, блок 7).

In [ ]:
CONDITIONS = "image_only no_proprio no_wrist full"
TASKS = " ".join(str(t) for t in range(10))
TRIALS = 3

!cd /content/openvla-oft && {PY} {SCRIPT} {ARGS} --conditions {CONDITIONS} --tasks {TASKS} --trials {TRIALS} --out {OUT}/series

## 4.1. Модель, обученная без камеры на запястье

Заморозка показывает, **на что опирается** обученная модель, но не отвечает, какой была бы модель,
**обученная без** входа. Для камеры на запястье такая модель есть:
`Sylvest/openvla-7b-oft-finetuned-libero-without-wrist` (авторы LIBERO-Plus) — **те же данные**
(432 демонстрации LIBERO-Spatial), тот же рецепт OFT, но одна внешняя камера + датчики.
Сравнение её `full` с `full` модели авторов — вклад камеры на запястье **при обучении**.
`no_proprio` — заморозка датчиков у неё (нужна ли ей живая информация о положении руки).
Скачивание ~15 ГБ, затем 2 × 30 попыток.

In [ ]:
!cd /content/openvla-oft && {PY} {SCRIPT} {ARGS} --model oft_no_wrist --conditions full no_proprio --tasks {TASKS} --trials {TRIALS} --out {OUT}/series_no_wrist

## 5. Результаты: успех и скорость

Для сравнения — OpenVLA из главы 1 на тех же 30 эпизодах: **76,7%** [59%, 88%], **0,83 с модели на каждый
шаг** (T4, 4 бита). Видеокарта здесь другая, поэтому для честного сравнения скорости смотрите и на
«запросов к модели на эпизод»: у OpenVLA модель спрашивали на каждом шаге.

In [ ]:
import json
import numpy as np
import pandas as pd

def wilson(k, n, z=1.96):
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return 100 * (c - h), 100 * (c + h)

# (папка, условие, подпись): модель авторов (2 камеры) и модель, обученная без камеры на запястье
RUNS = [("series", "image_only", "OFT: 1 фото (запястье и датчики заморожены)"),
        ("series", "no_proprio", "OFT: 2 фото, датчики заморожены"),
        ("series", "no_wrist", "OFT: 1 фото + датчики (запястье заморожено)"),
        ("series", "full", "OFT: 2 фото + датчики (как у авторов)"),
        ("series_no_wrist", "full", "OFT без запястья (обучена): 1 фото + датчики"),
        ("series_no_wrist", "no_proprio", "OFT без запястья (обучена): датчики заморожены")]
rows = []
for folder, c, label in RUNS:
    path = f"{OUT}/{folder}/{c}/summary.csv"
    if not os.path.exists(path):
        continue
    d = pd.read_csv(path)
    k, n = int(d.success.sum()), len(d)
    lo, hi = wilson(k, n)
    rows.append({"условие": label, "успех": f"{100 * k / n:.1f}% ({k}/{n})", "95% ДИ": f"[{lo:.0f}%, {hi:.0f}%]",
                 "шагов (сред.)": round(d.steps.mean()), "запросов к модели": round(d.queries.mean(), 1),
                 "с на запрос": round(d.infer_s_per_query.mean(), 3), "с модели на шаг": round(d.infer_s_per_step.mean(), 3),
                 "с симулятора на шаг": round(d.sim_s_per_step.mean(), 3), "с на эпизод": round(d.wall_s.mean(), 1)})
rows.append({"условие": "OpenVLA (глава 1)", "успех": "76.7% (23/30)", "95% ДИ": "[59%, 88%]",
             "с модели на шаг": 0.83, "запросов к модели": "каждый шаг"})
display(pd.DataFrame(rows))

## 6. Типы неудач

Те же правила, что в главе 1: «замирание» (захват не прошёл 5 см к 150-му шагу), «пустой захват»
(сжимал захват, но не коснулся миски), «донёс, но не поставил», «столкновение» — и новый тип
«ни разу не сжал захват» (робот двигался, но так и не попытался взять миску).

In [ ]:
TARGET = "akita_black_bowl_1"

def failure_type(log):
    eef = log[["eef_x", "eef_y", "eef_z"]].to_numpy()
    moved = np.flatnonzero(np.cumsum(np.linalg.norm(np.diff(eef, axis=0), axis=1)) > 0.05)
    if not len(moved) or moved[0] > 150:
        return "замирание"
    if log[f"{TARGET}_z"].max() - log[f"{TARGET}_z"].iloc[0] > 0.05:
        return "донёс, но не поставил"
    if (log.act_env_6 > 0).sum() == 0:
        return "ни разу не сжал захват"
    touch = (log[f"force_{TARGET}"] > 1).sum()
    other = (log.force_other > 1).sum()
    if (log.act_env_6 > 0).sum() > 0 and touch == 0 and other < 10:
        return "пустой захват"
    return "столкновение" if other >= 10 else "другое"

table = {}
for folder, c, label in RUNS:
    path = f"{OUT}/{folder}/{c}/summary.csv"
    if not os.path.exists(path):
        continue
    d = pd.read_csv(path)
    types = [failure_type(pd.read_csv(f"{OUT}/{folder}/{c}/{SUITE}_task{r.task_id:02d}_trial{r.trial:02d}.csv"))
             for _, r in d[~d.success].iterrows()]
    table[label] = pd.Series(types, dtype=object).value_counts()
table["OpenVLA (глава 1)"] = pd.Series({"замирание": 3, "пустой захват": 2, "столкновение": 1, "донёс, но не поставил": 1})
display(pd.DataFrame(table).fillna(0).astype(int))

## 6.1. Опыт с коробкой печенья (по желанию)

Та же команда, что в главе 1, — теперь для OFT с обеими камерами и датчиками.

In [ ]:
MY_COMMAND = "pick up the cookie box and place it on the plate"
!cd /content/openvla-oft && {PY} {SCRIPT} {ARGS} --conditions full --tasks 2 --trials 1 --instruction "{MY_COMMAND}" --goal-on cookies_1 plate_1 --out {OUT}/cookie
display(Video(f"{OUT}/cookie/full/{SUITE}_task02_trial00.mp4", embed=True, width=768))

## 7. Сохранение результатов

Сессия Colab стирается при закрытии — скачайте архив и положите в репозиторий в `results/`.

In [ ]:
# На Google Диск:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/vla-results && cp -r /content/results/* /content/drive/MyDrive/vla-results/

In [ ]:
# Или архивом на компьютер:
!cd /content && zip -qr results_oft.zip results
from google.colab import files
files.download("/content/results_oft.zip")